# 내 프로젝트 v1 : 설비 고장 데이터

# 실습 16 - 문제 정의서

## Step 0. 문제 정의

**한 문장** : [설비의 측정값으로, 오늘 가동하는 설비 중 고장 위험이 높은 순서를 매긴다]

| 물음 | 내 답 |
|---|---|
| 누가 쓰나 | [생산 라인 담당자] |
| 무엇을 결정하나 | [오늘 어떤 설비부터 순회 점검할지 순서] |
| 언제 결정하나 | [매일 가동 시작 전, 그날의 측정값이 모였을 때] |
| 이 판단이 늦으면 | [우선순위기 낮은 설비부터 보다가, 정작 위험한 설비의 고장을 늦게 발견한다.] |

## Step 1. 데이터 불러오기

In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\정유진\Desktop\secom-project\data\16_machine-failure.csv")

print("전체 행 수:", len(df))
print("열 개수:", df.shape[1])

숫자열개수 = len(df.select_dtypes(include="number").columns)
print("숫자로 된 열 개수:", 숫자열개수)

# 이 데이터셋의 결과 열은 "result"가 아니라 "Machine failure"이다
print()
print("Machine failure 값별 건수:")
print(df["Machine failure"].value_counts())
print()
print("Machine failure 값별 비율(%):")
print(round(df["Machine failure"].value_counts(normalize=True) * 100, 2))


전체 행 수: 10000
열 개수: 7
숫자로 된 열 개수: 5

Machine failure 값별 건수:
Machine failure
정상    9661
고장     339
Name: count, dtype: int64

Machine failure 값별 비율(%):
Machine failure
정상    96.61
고장     3.39
Name: proportion, dtype: float64


### 쓰는 데이터

| 항목 | 내용 |
|---|---|
| 출처 | [공개 데이터 (수업에서 받은 정제본)] |
| 규모 | [10000행 x 7열] |
| 입력으로 쓸 열 | [숫자로 된 측정 열 5개] |
| 결과 열 | [Machine failure] |
| 결과 열의 값 | [정상 / 고장] |
| 기준선이 될 숫자 | [고장 339건 (3.39%)] |

## Step 2. 기준 모델과 로지스틱 회귀

In [2]:
import pandas as pd  # 표를 다루는 도구
from sklearn.model_selection import train_test_split  # 학습용/시험용을 나누는 도구

df = pd.read_csv("../../data/16_machine-failure.csv")  # 데이터를 불러온다

숫자열 = df.select_dtypes(include="number").columns.tolist()  # 숫자로 된 열 이름만 뽑는다
X = df[숫자열]  # 숫자 열만 입력으로 쓴다 (결과 열은 문자열이라 여기서 자연히 빠진다)

df["label"] = (df["Machine failure"] == "고장").astype(int)  # 고장이면 1, 정상이면 0
y = df["label"]  # 정답 열

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 학습용 8, 시험용 2로 나눈다
    random_state=42,    # 다시 돌려도 같은 결과가 나오게 고정
    stratify=y           # 학습용/시험용의 1(고장) 비율을 같게 맞춘다
)

print("학습용:", X_train.shape, "고장 비율:", round(y_train.mean() * 100, 2), "%")  # 나눈 결과 확인
print("시험용:", X_test.shape, "고장 비율:", round(y_test.mean() * 100, 2), "%")  # 나눈 결과 확인


학습용: (8000, 5) 고장 비율: 3.39 %
시험용: (2000, 5) 고장 비율: 3.4 %


In [3]:
import numpy as np  # 기준 모델 답안지를 만드는 데 쓴다
from sklearn.preprocessing import StandardScaler  # 자릿수를 맞추는 도구
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 모델
from sklearn.metrics import accuracy_score  # 정확도를 재는 도구

기준예측 = np.zeros(len(y_test), dtype=int)  # 학습 없이 시험용 전부를 0(정상)이라 답한다
기준정확도 = accuracy_score(y_test, 기준예측)  # 기준 모델의 정확도

스케일러 = StandardScaler()  # 표준화 도구를 만든다
X_train_스케일 = 스케일러.fit_transform(X_train)  # 학습용 기준으로 표준화를 배우고 적용한다
X_test_스케일 = 스케일러.transform(X_test)  # 같은 기준으로 시험용도 바꾼다 (학습에는 안 씀)

모델 = LogisticRegression()  # 로지스틱 회귀 모델을 만든다
모델.fit(X_train_스케일, y_train)  # 학습용으로만 학습시킨다

print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")  # 학습 전 미리 확인


기준 모델 정확도: 96.6 %


In [4]:
from sklearn.metrics import precision_score, recall_score  # 정밀도·재현율을 재는 도구

예측 = 모델.predict(X_test_스케일)  # 시험용 입력만 넣어 답을 받는다 (시험용은 학습에 쓰지 않았다)

정확도 = accuracy_score(y_test, 예측)  # 내 모델의 정확도
지목건수 = int((예측 == 1).sum())  # 고장이라고 지목한 건수
지목중진짜 = int(((예측 == 1) & (y_test == 1)).sum())  # 그중 실제로 고장이었던 건수
정밀도 = precision_score(y_test, 예측, zero_division=0)  # 지목한 것 중 진짜였던 비율
재현율 = recall_score(y_test, 예측, zero_division=0)  # 실제 고장 중 잡아낸 비율

print("1. 기준 모델 정확도:", round(기준정확도 * 100, 2), "%")  # 학습 없이 찍은 점수
print("2. 로지스틱회귀모델(모델_v1) 정확도:", round(정확도 * 100, 2), "%")  # 학습시킨 모델의 점수
print("3. 지목한 건수:", 지목건수)  # 모델이 고장이라 한 건수
print("4. 그중 진짜:", 지목중진짜)  # 지목한 것 중 실제 고장 건수
print("5. 시험용의 실제 건수:", int(y_test.sum()))  # 시험용 안의 진짜 고장 건수
print("6. 정밀도:", round(정밀도, 3))  # 지목 중 진짜였던 비율
print("7. 재현율:", round(재현율, 3))  # 실제 고장 중 잡아낸 비율


1. 기준 모델 정확도: 96.6 %
2. 로지스틱회귀모델(모델_v1) 정확도: 96.85 %
3. 지목한 건수: 13
4. 그중 진짜: 9
5. 시험용의 실제 건수: 68
6. 정밀도: 0.692
7. 재현율: 0.132


In [5]:
# 앞 모델(모델)은 그대로 두고, class_weight="balanced"만 추가한 모델을 새 이름으로 만든다
가중치모델 = LogisticRegression(class_weight="balanced")  # 드문 쪽(고장)을 무겁게 세도록 설정
가중치모델.fit(X_train_스케일, y_train)  # 학습용으로만 학습시킨다

가중치예측 = 가중치모델.predict(X_test_스케일)  # 시험용 입력만 넣어 답을 받는다

가중치지목건수 = int((가중치예측 == 1).sum())  # 가중치 모델이 고장이라 한 건수
가중치지목중진짜 = int(((가중치예측 == 1) & (y_test == 1)).sum())  # 그중 실제 고장 건수
가중치재현율 = recall_score(y_test, 가중치예측, zero_division=0)  # 실제 고장 중 잡아낸 비율

print("[로지스틱회귀모델(모델_v1)] 지목 건수:", 지목건수, "/ 그중 진짜:", 지목중진짜, "/ 재현율:", round(재현율, 3))
print("[가중치 모델]   지목 건수:", 가중치지목건수, "/ 그중 진짜:", 가중치지목중진짜, "/ 재현율:", round(가중치재현율, 3))


[로지스틱회귀모델(모델_v1)] 지목 건수: 13 / 그중 진짜: 9 / 재현율: 0.132
[가중치 모델]   지목 건수: 404 / 그중 진짜: 57 / 재현율: 0.838


In [6]:
가중치정확도 = accuracy_score(y_test, 가중치예측)  # 가중치 모델의 정확도
print("가중치 모델 정확도:", round(가중치정확도 * 100, 2), "%")


가중치 모델 정확도: 82.1 %


### [도전 - 하이퍼파라미터를 바꿔본 결과]

| | 지목 | 그중 진짜 | 재현율 |
|---|---|---|---|
| 기준 모델 | 0 | 0 | 0 |
| 로지스틱회귀모델 | 13 | 9 | 0.132 |
| 가중치 모델 | 404 | 57 | 0.838 |

- 알게 된 것 : 기준 모델은 고장을 아예 지목하지 않아 재현율이 0이었다. 로지스틱회귀모델은 놓치는 걸 크게 줄이지는 못했고(재현율 0.132), 가중치 모델로 바꾸자 놓친 것은 크게 줄었지만(재현율 0.838) 헛걸음이 13건에서 404건으로 늘었다. 어느 쪽이 나은지는 재검사 한 건에 드는 비용을 알아야 정해진다

## Step 3. 한계

- [실제 고장의 상당수를 놓친다 — 기본 모델(로지스틱 회귀 모델)은 실제 고장 68건 중 9건만 잡아냈고(재현율 0.132), class_weight를 조정해도 재현율은 오르는 대신 헛걸음이 크게 늘어나는 맞교환만 확인했을 뿐, 실제로 현장에 쓸 만한 균형점은 아직 못 정했다]
- [왜 고장 위험이 높다고 판단했는지 이유를 보여주지 않는다 — 어떤 측정값(공정온도, 공구마모 등)이 판단에 크게 작용했는지는 오늘 확인하지 않아서, "무엇 때문에"라는 질문에는 답할 수 없다]
- [이 결과가 다른 시기·다른 설비에도 똑같이 통할지는 검증되지 않았다 — 지금 가진 10,000건 데이터 안에서만 확인한 결과라, 계절 변화나 다른 설비 유형에 적용했을 때도 같은 성능이 나올지는 알 수 없다]

### [포트폴리오 카드 - 설비 고장 예측 (v1)]

**문제** : 설비의 측정값(공기온도·공정온도·회전속도·토크·공구마모)으로, 오늘 가동하는 설비 중 고장 위험이 높은 순서를 매긴다

| 물음 | 내 답 |
|---|---|
| 누가 쓰나 | 생산 라인 관리자 |
| 무엇을 결정하나 | 오늘 어떤 설비부터 순회 점검할지 순서 |
| 언제 결정하나 | 매일 가동 시작 전, 그날의 측정값이 모였을 때 |
| 이 판단이 늦으면 | 우선순위가 낮은 설비부터 보다가, 정작 위험한 설비의 고장을 늦게 발견한다 |

**모델 비교**

| | 정확도 | 정밀도 | 재현율 | 지목 | 그중 진짜 | 핵심 요약 |
|---|---|---|---|---|---|---|
| 기준 모델 | 96.6% | - (아예 지목 안 함) | 0 | 0 | 0 | 전부 정상이라고만 답해, 고장을 하나도 잡아내지 못함 |
| 로지스틱회귀모델 | 96.85% | 0.692 | 0.132 | 13 | 9 | 지목한 13건 중 9건은 맞았지만, 실제 고장 68건 중 9건만 잡아냄 |
| 가중치 모델 | 82.1% | 0.141 | 0.838 | 404 | 57 | 지목을 404건으로 크게 늘려 실제 고장 68건 중 57건을 잡아냈지만, 헛걸음도 13건에서 404건으로 크게 늘어남 |

**한 줄 요약** : 정확도는 기준 모델과 비슷하지만, 로지스틱회귀모델 혼자서는 실제 고장을 잡아내는 능력(재현율)이 낮았다. class_weight="balanced"를 넣은 가중치 모델은 재현율을 0.838까지 크게 올렸지만, 그만큼 헛걸음(정밀도 하락)도 늘어나는 맞교환이 확인됐다.

**한계**
1. 실제 고장의 상당수를 놓친다 — 정밀도와 재현율 중 어디에 무게를 둘지 아직 정하지 못했다
2. 왜 고장 위험이 높다고 판단했는지 이유(어떤 측정값이 크게 작용했는지)를 보여주지 않는다
3. 지금 가진 데이터 안에서만 확인한 결과라, 다른 시기·다른 설비에도 똑같이 통할지는 검증되지 않았다

**다음에 할 일**
1. 재검사 한 건에 드는 비용을 조사해서, 정밀도와 재현율 중 어디에 무게를 둘지 정한다
2. 로지스틱 회귀 말고 다른 모델(랜덤포레스트 등)과도 비교해본다
3. 어떤 측정값이 고장 판단에 가장 크게 작용했는지 확인한다

# 실습 17 - 모델링 계획표

## STEP 1. 모델링 계획

| # | 하는 일 | 내 경우엔 | 막힐 것 같나 |
|---|---|---|---|
| 1 | 파일 불러오기 | ../../data/16_machine-failure.csv | |
| 2 | 빈칸 채우기 | 결측치 없음(확인됨) - 채울 빈칸 없음 | |
| 3 | 결과 열을 숫자로 | Machine failure가 "고장"이면 1, "정상"이면 0 | |
| 4 | 입력과 정답 나누기 | 입력은 숫자 열 5개(공기온도·공정온도·회전속도·토크·공구마모), 정답은 고장 여부(0/1) | |
| 5 | 학습용·시험용 나누기 | 8대 2, 고장 비율(3.39%)을 양쪽에 맞춰서(stratify) | 막힐 것 같음 |
| 6 | 기준 모델 점수 재기 | 전부 정상(0)이라 답하는 모델의 정확도 | |
| 7 | 모델 하나 학습시키기 | 표준화한 뒤 로지스틱 회귀 | 막힐 것 같음 |
| 8 | 결과 읽기 | 기준 모델과 나란히 놓고 정확도·정밀도·재현율 비교 | |

## STEP 2. 무슨 숫자를 볼지 정하기
### 이번 v1에서 볼 숫자

- 볼 지표 : [기준 모델 정확도 / 정확도 accuracy / 정밀도 precision / 재현율 recall]
- 이걸 보는 이유 : [불량이 3.3%라 전부 양품이라 답해도 96%가 넘는다. 정확도만 보면 잘한 걸로 착각한다]

저장소에 올린 날 : [2026-08-24]

### [도전 - 입력에서 빼야 할 열이 있나 (데이터 누수 점검)]

| 종류 | 있나 | 열 이름 | 어떻게 할까 |
|---|---|---|---|
| 번호·순번 열 | [없음] | [-] | [-] |
| 판단 시점에 아직 없는 열 | [없음] | [-] | [-] |
| 결과 열의 사본 | [없음] | [-] | [-] |

- 결론 : [입력 5개를 그대로 쓴다. 다만 정확도가 99%를 넘으면 이 표를 다시 본다]

# 실습 18 - 모델링 v1

## 모델링 v1

### STEP 1. 문법 노트 - 코드에서 찾을 이름

| 이름 | 뜻 |
|---|---|
| features | 모델에 넣는 입력 열 목록 |
| label | 정답표 - 찾으려는 쪽이면 1, 아니면 0 |
| X / y | 입력 / 정답 |
| X_train, X_test | 학습용 입력 / 시험용 입력 |
| y_train, y_test | 학습용 정답 / 시험용 정답 |
| baseline | 기준 모델 - 학습 없이 전부 아니라고 답하는 것 |
| model | 학습시킨 모델 |
| y_pred | 모델이 내놓은 답 |
| precision / recall | 정밀도 / 재현율 |

### Step 2: 내 갈래 정하기

- 갈래 : [A - 맞히기]
- 결과 열 : [Machine failure] / 관심 갈래 : [고장]
- 이 기준으로 정한 이유 : [오늘 가동하는 설비 중 고장 위험이 높은 순서를 매기기 위해서]

### Step 3: 여덟 줄을 그대로 시키기 

In [7]:
import pandas as pd  # 표를 다루는 도구
from sklearn.model_selection import train_test_split  # 학습용/시험용을 나누는 도구

df = pd.read_csv("../../data/16_machine-failure.csv")  # 1. 데이터를 불러온다

숫자열 = df.select_dtypes(include="number").columns.tolist()  # 숫자로 된 열 이름만 뽑는다
X = df[숫자열].copy()  # 2. 결과 열(Machine failure)은 답이므로 여기서 자연히 빠진다

X = X.fillna(X.median())  # 3. 입력 열의 빈칸을 그 열의 중앙값으로 채운다

df["label"] = (df["Machine failure"] == "고장").astype(int)  # 4. 고장이면 1, 아니면 0
y = df["label"]  # 정답 열

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 5. 학습용 8, 시험용 2로 나눈다
    random_state=42,    # 다시 돌려도 같은 결과가 나오게 고정
    stratify=y           # 학습용/시험용의 1(고장) 비율을 같게 맞춘다
)

print("학습용:", X_train.shape, "고장 비율:", round(y_train.mean() * 100, 2), "%")  # 나눈 결과 확인
print("시험용:", X_test.shape, "고장 비율:", round(y_test.mean() * 100, 2), "%")  # 나눈 결과 확인


학습용: (8000, 5) 고장 비율: 3.39 %
시험용: (2000, 5) 고장 비율: 3.4 %


In [8]:
# label(정답 열, y) 확인 - 실제로 만들어진 값을 눈으로 본다
print("label 값별 건수:")
print(y.value_counts())
print()
print("label 값별 비율(%):")
print(round(y.value_counts(normalize=True) * 100, 2))
print()

# 검산 - 원본 Machine failure 값과 견줘 잘 바뀌었는지 확인
print("검산 - label==1(고장) 개수와 원본 '고장' 건수가 같은가:",
      y.sum() == (df["Machine failure"] == "고장").sum())


label 값별 건수:
label
0    9661
1     339
Name: count, dtype: int64

label 값별 비율(%):
label
0    96.61
1     3.39
Name: proportion, dtype: float64

검산 - label==1(고장) 개수와 원본 '고장' 건수가 같은가: True


In [9]:
import numpy as np  # 기준 모델 답안지를 만드는 데 쓴다
from sklearn.preprocessing import StandardScaler  # 자릿수를 맞추는 도구
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 모델
from sklearn.metrics import accuracy_score  # 정확도를 재는 도구

기준예측 = np.zeros(len(y_test), dtype=int)  # 6. 학습 없이 시험용 전부를 0(정상)이라 답한다
기준정확도 = accuracy_score(y_test, 기준예측)  # 기준 모델의 정확도

스케일러 = StandardScaler()  # 7. 표준화 도구를 만든다
X_train_스케일 = 스케일러.fit_transform(X_train)  # 학습용 기준으로 표준화를 배우고 적용한다
X_test_스케일 = 스케일러.transform(X_test)  # 같은 기준으로 시험용도 바꾼다 (학습에는 안 씀)

모델_v2 = LogisticRegression()  # 로지스틱 회귀 모델을 만든다 (앞의 모델과 구분되는 새 이름)
모델_v2.fit(X_train_스케일, y_train)  # 학습용으로만 학습시킨다

print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")  # 학습 전 미리 확인


기준 모델 정확도: 96.6 %


In [10]:
from sklearn.metrics import precision_score, recall_score  # 정밀도·재현율을 재는 도구

예측_v2 = 모델_v2.predict(X_test_스케일)  # 8. 시험용 입력만 넣어 답을 받는다 (시험용은 학습에 쓰지 않았다)

정확도_v2 = accuracy_score(y_test, 예측_v2)  # 내 모델의 정확도
지목건수_v2 = int((예측_v2 == 1).sum())  # 고장이라고 지목한 건수
지목중진짜_v2 = int(((예측_v2 == 1) & (y_test == 1)).sum())  # 그중 실제로 고장이었던 건수
정밀도_v2 = precision_score(y_test, 예측_v2, zero_division=0)  # 지목한 것 중 진짜였던 비율
재현율_v2 = recall_score(y_test, 예측_v2, zero_division=0)  # 실제 고장 중 잡아낸 비율

print("1. 기준 모델 정확도:", round(기준정확도 * 100, 2), "%")  # 학습 없이 찍은 점수
print("2. 내 모델 정확도:", round(정확도_v2 * 100, 2), "%")  # 학습시킨 모델의 점수
print("3. 지목한 건수:", 지목건수_v2)  # 모델이 고장이라 한 건수
print("4. 그중 진짜:", 지목중진짜_v2)  # 지목한 것 중 실제 고장 건수
print("5. 시험용의 실제 건수:", int(y_test.sum()))  # 시험용 안의 진짜 고장 건수
print("6. 정밀도:", round(정밀도_v2, 3))  # 지목 중 진짜였던 비율
print("7. 재현율:", round(재현율_v2, 3))  # 실제 고장 중 잡아낸 비율


1. 기준 모델 정확도: 96.6 %
2. 내 모델 정확도: 96.85 %
3. 지목한 건수: 13
4. 그중 진짜: 9
5. 시험용의 실제 건수: 68
6. 정밀도: 0.692
7. 재현율: 0.132


### STEP 4. 받은 코드 확인

| # | 확인한 것 | 어땠나 |
|---|---|---|
| 1 | stratify=y | 들어 있었다 |
| 2 | 표준화를 학습용에서만 fit | 들어 있었다 |
| 3 | 기준 모델 점수 | 들어 있었다 |
| 4 | 지목 건수·정밀도·재현율 | 들어 있었다 |

### STEP 5. 돌려서 숫자 읽기

| # | 항목 | 값 |
|---|---|---|
| 1 | 기준 모델 정확도 | 96.6 % |
| 2 | 내 모델 정확도 | 96.85 % |
| 3 | 지목한 건수 | 13 건 |
| 4 | 그중 진짜 | 9 건 |
| 5 | 시험용의 실제 | 68 건 |
| 6 | 정밀도 precision | 0.692 |
| 7 | 재현율 recall | 0.132 |

[오류 시 프롬프트]

@project_v1.ipynb 를 실행했더니 아래 오류가 났어.
무슨 뜻인지 먼저 한 줄로 알려주고, 어디를 고치면 되는지 알려줘.

(여기에 빨간 글씨를 처음부터 끝까지 그대로 붙여넣기)

내 데이터에 실제로 있는 열 이름을 확인해서 답해줘.
지어내지 말고 실제 실행값만 보여줘.

#### 직접 해보기(도전)

In [11]:
# 앞 모델(모델_v2)은 그대로 두고, class_weight="balanced"만 추가한 모델을 새 이름으로 만든다
가중치모델_v2 = LogisticRegression(class_weight="balanced")  # 드문 쪽(고장)을 무겁게 세도록 설정
가중치모델_v2.fit(X_train_스케일, y_train)  # 학습용으로만 학습시킨다

가중치예측_v2 = 가중치모델_v2.predict(X_test_스케일)  # 시험용 입력만 넣어 답을 받는다

가중치지목건수_v2 = int((가중치예측_v2 == 1).sum())  # 가중치 모델이 고장이라 한 건수
가중치지목중진짜_v2 = int(((가중치예측_v2 == 1) & (y_test == 1)).sum())  # 그중 실제 고장 건수
가중치재현율_v2 = recall_score(y_test, 가중치예측_v2, zero_division=0)  # 실제 고장 중 잡아낸 비율

print("[모델_v2]       지목 건수:", 지목건수_v2, "/ 그중 진짜:", 지목중진짜_v2, "/ 재현율:", round(재현율_v2, 3))
print("[가중치모델_v2] 지목 건수:", 가중치지목건수_v2, "/ 그중 진짜:", 가중치지목중진짜_v2, "/ 재현율:", round(가중치재현율_v2, 3))


[모델_v2]       지목 건수: 13 / 그중 진짜: 9 / 재현율: 0.132
[가중치모델_v2] 지목 건수: 404 / 그중 진짜: 57 / 재현율: 0.838


### [도전 - 하이퍼파라미터를 바꿔본 결과]

| | 지목 | 그중 진짜 | 재현율 |
|---|---|---|---|
| 손 안 댐 | [13] | [9] | [0.132] |
| 바꾼 뒤 | [404] | [57] | [0.838] |

- 알게 된 것 : [놓친 것은 줄었지만 헛걸음이 4건에서 347건으로 늘었다. 어느 쪽이 나은지는 재검사 한 건에 드는 비용을 알아야 정해진다]

In [10]:
# y(정답 열) 값별 건수 확인
print(y.value_counts())

# y 값별 비율(%) 확인
print(round(y.value_counts(normalize=True) * 100, 2))

# 검산: 1(고장) 개수가 원본 Machine failure의 "고장" 건수와 같은지
print(y.sum() == (df["Machine failure"] == "고장").sum())

label
0    9661
1     339
Name: count, dtype: int64
label
0    96.61
1     3.39
Name: proportion, dtype: float64
True


In [13]:
df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,label
0,M,298.1,308.6,1551,42.8,0,정상,0
1,L,298.2,308.7,1408,46.3,3,정상,0
2,L,298.1,308.5,1498,49.4,5,정상,0
3,L,298.2,308.6,1433,39.5,7,정상,0
4,L,298.2,308.7,1408,40.0,9,정상,0


# 실습 19 - 결과 한 장

## v1 결과

### STEP 1. 결과 한 장 만들기

| | 정확도 | 지목 | 그중 진짜 | 재현율 |
|---|---|---|---|---|
| 기준 모델 (전부 정상) | 96.60% | 0건 | 0건 | 0.000 |
| 내 모델 v1 | 96.85% | 13건 | 9건 | 0.132 |

- 시험용에 실제로 있던 고장 : 68건
- 한 줄 요약 : 정확도는 기준보다 0.25%p 올랐고, 지목한 13건 중 9건이 실제 고장이었다. 다만 실제 고장 68건 중 59건은 못 잡았다

### STEP 2. 다음에 할 일

1. [드문 쪽을 무겁게 세는 설정으로 지목 건수를 늘려보기]
2. [지목을 늘렸을 때 헛걸음이 얼마나 느는지 같이 재기]
3. [재검사 한 건에 드는 비용을 알아보고, 어느 쪽이 나은지 정하기]

### Step 3: 포트폴리오 카드 두 칸 채우기

### 카드에 옮겨 적을 것

**3번 칸 - 사용 데이터**
[설비 고장 기록 / 10,000행 x 7열 / 결과 열은 정상·고장 / 고장 339건 (3.39%)]

**5번 칸 - 결과 수치**
[전부 정상이라 답하는 기준 모델 96.6% 대비 96.85%.
지목한 13건 중 9건이 실제 불합격이었으나(정밀도 0.692),
실제 68건 중 59건은 못 잡았다(재현율 0.132). 그래서 정확도만으로는 판단할 수 없는 문제였다]

**위 문장을 다듬고 싶을때 프롬프트**

@project_v1.ipynb 의 v1 결과 칸과 카드에 옮겨 적을 것 칸을 읽고,
아래 두 가지만 고쳐줘.

1. 원인이라고 단정한 문장이 있으면 "~에서 ~가 높게 관찰됨" 식으로 바꾸기
2. 읽는 사람이 이 분야를 모른다고 보고, 어려운 말이 있으면 뒤에 쉬운 말로 한 번 풀기

숫자는 하나도 바꾸지 마. 없는 내용을 새로 지어내지도 마.
고친 것을 노트북에 새 글 칸으로 넣어줘.

### Step 4: 갤러리에 올리기

### Step 5: 오늘 적어둔 것 모아서 일지로

- 오늘 정한 문제 : [설비 측정값으로 오늘 가동하는 설비 중 고장 위험이 높은 순서를 고르는 것]
- 쓴 데이터 : [셀 검사 기록 10,000행 x 7열 / 고장 339건 3.39%]
- 오늘 나온 숫자 : [기준 96.6% / 로지스틱스회귀모델(v1) 96.85% / 지목 13건 중 진짜 9건 / 재현율 0.132]
- 다음에 손댈 자리 : [지목 건수를 늘리는 설정]

### 도전 - 그래서 무엇을 하자는 것인가

- [모델이 고장 가능성이 높다고 지목한 설비는 오늘 순회 점검 순서에서 먼저 확인하자는 것.
지금 v1(로지스틱회귀모델)로는 실제 고장 68건 중 9건만 걸러지므로, 이 상태로 현장에 걸기에는 이르다]